In [1]:
import openai, os
import faiss
from llama_index.embeddings import LangchainEmbedding
from llama_index import (
    SimpleDirectoryReader, 
    ServiceContext,
    VectorStoreIndex,
    StorageContext
)

from llama_index.vector_stores.faiss import FaissVectorStore
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from llama_index.node_parser import SimpleNodeParser
openai.api_key = ""
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=100, chunk_overlap=20)
parser = SimpleNodeParser()

documents = SimpleDirectoryReader('./data/faq/').load_data()
nodes = parser.get_nodes_from_documents(documents)
embed_model = LangchainEmbedding(HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
))
service_context = ServiceContext.from_defaults(embed_model=embed_model)
dimension = 768
faiss_index = faiss.IndexFlatIP(dimension)

vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, service_context=service_context
)

/usr/local/anaconda3/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [2]:
openai.api_key = os.environ.get("OPENAI_API_KEY")
query_engine = index.as_query_engine(verbose=True)
response = query_engine.query("请问你们海南能发货吗")
print(response.response)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


我们支持海南省的配送。


In [3]:
response = query_engine.query("你们用哪些快递公司送货？")
print(response)

我们与顺丰速运、圆通速递、申通快递、韵达快递、中通快递、百世快递等多家知名快递公司合作。


In [4]:
response = query_engine.query("你们的退货政策是怎么样的？")
print(response)

自收到商品之日起7天内，如产品未使用、包装完好，您可以申请退货。某些特殊商品可能不支持退货，请在购买前查看商品详情页面的退货政策。


In [5]:
# %pip install icetk
# %pip install cpm_kernels

In [ ]:
from transformers import AutoTokenizer, AutoModel
# tokenizer = AutoTokenizer.from_pretrained("THUDM/chatglm-6b-int4", trust_remote_code=True)
# model = AutoModel.from_pretrained("THUDM/chatglm-6b-int4", trust_remote_code=True).half().cuda()
# model = AutoModel.from_pretrained("THUDM/chatglm-6b-int4",trust_remote_code=True).float()
tokenizer = AutoTokenizer.from_pretrained("THUDM/chatglm3-6b", trust_remote_code=True)
model = AutoModel.from_pretrained("THUDM/chatglm3-6b", trust_remote_code=True).float()
model = model.eval()

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [1]:
question = """
自收到商品之日起7天内，如产品未使用、包装完好，您可以申请退货。某些特殊商品可能不支持退货，请在购买前查看商品详情页面的退货政策。
根据以上信息，请回答下面的问题：
Q: 你们的退货政策是怎么样的？f
"""
response, history = model.chat(tokenizer, question, history=[])
print(response)

NameError: name 'model' is not defined

In [ ]:
question = """
Q: 你们的退货政策是怎么样的？
A: 
"""
response, history = model.chat(tokenizer, question, history=[])
print(response)

In [ ]:
import openai, os
import faiss
from llama_index.embeddings import LangchainEmbedding
from llama_index import (
    SimpleDirectoryReader, 
    ServiceContext,
    VectorStoreIndex,
    StorageContext
)
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from llama_index.node_parser import SimpleNodeParser
from langchain.llms.base import LLM
from llama_index import LLMPredictor
from typing import Optional, List, Mapping, Any
class CustomLLM(LLM):
    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        response, history = model.chat(tokenizer, prompt, history=[])
        return response
    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"name_of_model": "chatglm-6b-int4"}
    @property
    def _llm_type(self) -> str:
        return "custom"


# import faiss
# from llama_index.embeddings import LangchainEmbedding
# from llama_index import (
#     SimpleDirectoryReader, 
#     ServiceContext,
#     VectorStoreIndex,
#     StorageContext
# )

# from llama_index.vector_stores.faiss import FaissVectorStore
# from langchain.embeddings.huggingface import HuggingFaceEmbeddings
# from langchain.text_splitter import CharacterTextSplitter
# from llama_index.node_parser import SimpleNodeParser
# openai.api_key = ""
# text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=100, chunk_overlap=20)
# parser = SimpleNodeParser()

# documents = SimpleDirectoryReader('./data/faq/').load_data()
# nodes = parser.get_nodes_from_documents(documents)
# embed_model = LangchainEmbedding(HuggingFaceEmbeddings(
#     model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
# ))
# service_context = ServiceContext.from_defaults(embed_model=embed_model)
# dimension = 768
# faiss_index = faiss.IndexFlatIP(dimension)

# vector_store = FaissVectorStore(faiss_index=faiss_index)
# storage_context = StorageContext.from_defaults(vector_store=vector_store)
# index = VectorStoreIndex.from_documents(
#     documents, storage_context=storage_context, service_context=service_context
# )

In [ ]:
from langchain.text_splitter import SpacyTextSplitter
llm_predictor = LLMPredictor(llm=CustomLLM())
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=100, chunk_overlap=20)
parser = parser = SimpleNodeParser()
# documents = SimpleDirectoryReader('./drive/MyDrive/colab_data/faq/').load_data()
documents = SimpleDirectoryReader('./data/faq/').load_data()
# nodes = parser.get_nodes_from_documents(documents)
embed_model = LangchainEmbedding(HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
))
service_context = ServiceContext.from_defaults(embed_model=embed_model, llm_predictor=llm_predictor)
dimension = 768
faiss_index = faiss.IndexFlatIP(dimension)
# index = GPTFaissIndex(nodes=nodes, faiss_index=faiss_index, service_context=service_context)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, service_context=service_context
)
from llama_index import PromptTemplate
from llama_index import QueryMode
QA_PROMPT_TMPL = (
    "{context_str}"
    "\n\n"
    "根据以上信息，请回答下面的问题：\n"
    "Q: {query_str}\n"
    )
QA_PROMPT = PromptTemplate(QA_PROMPT_TMPL)
response = index.query(
    "请问你们海南能发货吗？", 
    mode=QueryMode.EMBEDDING,
    text_qa_template=QA_PROMPT,
    verbose=True, 
)
print(response)
